In [24]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import selfmadefunctions
import importlib
importlib.reload(selfmadefunctions)
import pandas as pd

# Author: Manou Liesker. Student number: 15250946

In [ ]:
def track_video(networkfolder,filename, show, save_video, save_csv):
    input_path = str(networkfolder / filename)

    cap = cv2.VideoCapture(input_path)

    if save_video:
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps <= 0:
            fps = 30

        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        output_path = str(networkfolder / f"{Path(filename).stem}_tracked.avi")
        fourcc = cv2.VideoWriter_fourcc(*"XVID")
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height), isColor=True)

    x_points = []
    y_points = []
    frame_numbers = []

    threshold_value = 100
    min_area = 3
    max_area = 100
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (5, 5), 0)

        _, mask = cv2.threshold(gray, threshold_value, 255, cv2.THRESH_BINARY)

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        small_contours = [
            cnt for cnt in contours
            if min_area <= cv2.contourArea(cnt) <= max_area
        ]

        frame_numbers.append(frame_idx)

        if small_contours:
            cnt = max(small_contours, key=cv2.contourArea)

            M = cv2.moments(cnt)
            if M["m00"] != 0:
                cx = M["m10"] / M["m00"]
                cy = M["m01"] / M["m00"]

                x_points.append(cx)
                y_points.append(cy)

                cv2.drawContours(frame, [cnt], -1, (0, 255, 0), 2)
                cv2.circle(frame, (int(cx), int(cy)), 5, (0, 0, 255), -1)
            else:
                x_points.append(np.nan)
                y_points.append(np.nan)
        else:
            x_points.append(np.nan)
            y_points.append(np.nan)

        if show:
            cv2.imshow("tracking", frame)
            cv2.imshow("mask", mask)
            if cv2.waitKey(30) == 27:
                break

        if save_video:
            out.write(frame)

        frame_idx += 1

    cap.release()

    if save_video:
        out.release()
        print("Saved video to:", output_path)

    if save_csv:
        lowest_y = np.nanmax(y_points)
        new_points = []

        for y in y_points:
            new_y = lowest_y - y if not np.isnan(y) else np.nan
            new_points.append(new_y)

        cleaned_file = pd.DataFrame({
            'Frame': frame_numbers,
            'Y': new_points
        })

        NETWORK_FOLDER = Path(r"Z:\Clean_Data\Data_Manou_Thesis_Clean")
        csv_path = NETWORK_FOLDER / f"{Path(filename).stem}_clean.csv"
        cleaned_file.to_csv(csv_path, index=False)
        print(f"Saved as {csv_path}")

    cv2.destroyAllWindows()

NETWORK_FOLDER = Path(r"Z:\Video_files\Test_Videos")
track_video("Reflection_Method_Test.avi", show=False, save_video=True, save_csv=True)

Saved video to: Z:\Video_Files\Reflection_Method_Test_tracked.avi
[20.669047619047618, 20.669047619047618, 20.669047619047618, 20.669047619047618, 20.725225225225223, 20.532374100719423, 20.767543859649123, 20.685185185185183, 20.781609195402297, 20.83449883449883, 20.624413145539904, 20.5947242206235, 20.925407925407924, 20.705215419501133, 20.58675799086758, 20.666666666666664, 20.715617715617714, 20.830985915492956, 20.62192393736018, 20.778554778554778, 20.63982102908277, 20.796296296296294, 20.78747203579418, 20.524475524475523, 20.784580498866212, 20.83449883449883, 20.533783783783782, 20.736596736596734, 20.884976525821596, 20.699074074074073, 20.456521739130434, 20.654846335697396, 20.654846335697396, 20.65990990990991, 20.66435185185185, 20.66435185185185, 20.395522388059703, 20.736596736596734, 20.790286975717436, 20.764172335600904, 20.764172335600904, 20.754022988505746, 20.493150684931507, 20.67149758454106, 20.588607594936708, 20.736596736596734, 20.61737089201878, 20.537

In [ ]:
df = pd.read_csv("Reflection_Method_Test.avi_clean.csv")

x = df.iloc[:, 0]
y = df.iloc[:, 1]
yerr = df.iloc[:, 2]

plt.errorbar(x, y, yerr=yerr, fmt='o', capsize=4)
plt.xlabel("x")
plt.ylabel("y")
plt.show()

Saved video to: Z:\Video_Files\Reflection_Method_Test_tracked.avi
Number of frames: 403
Number of x points: 403
Number of y points: 403
